# GeoAI Aquaculture Pond Challenge — v2

**Key findings from EDA:**
- VH backscatter: ponds=-29.16 dB, non-ponds=-23.05 dB → 6 dB gap is huge
- No missing data in train; test has some -9999 cloud gaps
- 40% positive rate (735 ponds / 1821 total) — not severely imbalanced

**Strategy:** SAR-heavy feature engineering + temporal stability + LGB/CatBoost ensemble

In [ ]:
!pip install lightgbm catboost xgboost --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

np.random.seed(42)
MONTHS = [f'{i:02d}' for i in range(1, 13)]
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

## 1. Load & Clean

In [ ]:
DATA_DIR = './'

train = pd.read_csv(DATA_DIR + 'Train.csv')
test  = pd.read_csv(DATA_DIR + 'Test.csv')
sub   = pd.read_csv(DATA_DIR + 'SampleSubmission.csv')

# Replace -9999 sentinel with NaN
for df in [train, test]:
    df.replace(-9999,   np.nan, inplace=True)
    df.replace(-9999.0, np.nan, inplace=True)

print(f'Train: {train.shape} | Positive rate: {train["label"].mean():.2%}')
print(f'Test:  {test.shape}')

## 2. Feature Engineering

In [ ]:
def temporal_stats(series_list, prefix):
    """Compute temporal statistics across months from a list of Series."""
    ts = pd.concat(series_list, axis=1)
    n  = ts.notna().sum(axis=1).clip(lower=1)
    mean = ts.mean(axis=1)
    std  = ts.std(axis=1)
    return {
        f'{prefix}_mean':   mean,
        f'{prefix}_std':    std,
        f'{prefix}_min':    ts.min(axis=1),
        f'{prefix}_max':    ts.max(axis=1),
        f'{prefix}_range':  ts.max(axis=1) - ts.min(axis=1),
        f'{prefix}_median': ts.median(axis=1),
        f'{prefix}_p25':    ts.quantile(0.25, axis=1),
        f'{prefix}_p75':    ts.quantile(0.75, axis=1),
        # Coefficient of variation — stability across time (low = stable water)
        f'{prefix}_cv':     std / (mean.abs() + 1e-6),
        f'{prefix}_n_obs':  ts.notna().sum(axis=1),
    }


def temporal_trend(series_list, prefix):
    """Linear trend slope across months (positive = increasing over year)."""
    ts = pd.concat(series_list, axis=1).values.astype(float)
    x  = np.arange(ts.shape[1], dtype=float)
    # vectorised OLS slope
    x_dm = x - x.mean()
    slopes = np.nansum(ts * x_dm[None, :], axis=1) / (np.nansum(x_dm**2) + 1e-9)
    return {f'{prefix}_trend': pd.Series(slopes, dtype=float)}


def build_features(df):
    eps = 1e-6
    feats = {}

    # ── Per-month indices ─────────────────────────────────────────────────────
    ndwi_list, mndwi_list, ndvi_list, awei_list, awein_list = [], [], [], [], []
    sar_ratio_list, sar_diff_list, vh_list, vv_list = [], [], [], []
    rvi_list, ndwi2_list = [], []

    for m in MONTHS:
        g  = df[f'green_{m}']
        b  = df[f'blue_{m}']
        r  = df[f'red_{m}']
        ni = df[f'nir_{m}']
        s1 = df[f'swir1_{m}']
        s2 = df[f'swir2_{m}']
        vh = df[f'VH_{m}']
        vv = df[f'VV_{m}']

        # Water indices (higher = more water-like)
        ndwi  = (g - ni)  / (g + ni + eps)          # McFeeters NDWI
        mndwi = (g - s1)  / (g + s1 + eps)          # Xu MNDWI — better in urban
        ndwi2 = (ni - s1) / (ni + s1 + eps)         # Gao NDWI
        # Automated Water Extraction Index (removes shadow noise)
        awein = 4*(g - s1) - (0.25*ni + 2.75*s2)    # AWEInsh
        awei  = b + 2.5*g - 1.5*(ni + s1) - 0.25*s2 # AWEIsh
        ndvi  = (ni - r)  / (ni + r + eps)
        # Radar Vegetation Index (high = vegetation, low = water)
        rvi   = (4 * (10**(vh/10))) / ((10**(vh/10)) + (10**(vv/10)) + eps)

        # SAR features (in dB scale — keep as-is)
        sar_ratio = vh / (vv + eps)
        sar_diff  = vh - vv

        # Store per-month features
        feats[f'NDWI_{m}']     = ndwi
        feats[f'MNDWI_{m}']    = mndwi
        feats[f'NDWI2_{m}']    = ndwi2
        feats[f'NDVI_{m}']     = ndvi
        feats[f'AWEI_{m}']     = awei
        feats[f'AWEIn_{m}']    = awein
        feats[f'RVI_{m}']      = rvi
        feats[f'SAR_ratio_{m}']= sar_ratio
        feats[f'SAR_diff_{m}'] = sar_diff

        ndwi_list.append(ndwi);   mndwi_list.append(mndwi)
        ndvi_list.append(ndvi);   awei_list.append(awei)
        awein_list.append(awein); rvi_list.append(rvi)
        ndwi2_list.append(ndwi2)
        sar_ratio_list.append(sar_ratio); sar_diff_list.append(sar_diff)
        vh_list.append(vh);  vv_list.append(vv)

    # ── Temporal statistics for each derived index ────────────────────────────
    for name, lst in [
        ('NDWI',      ndwi_list),
        ('MNDWI',     mndwi_list),
        ('NDWI2',     ndwi2_list),
        ('NDVI',      ndvi_list),
        ('AWEI',      awei_list),
        ('AWEIn',     awein_list),
        ('RVI',       rvi_list),
        ('SAR_ratio', sar_ratio_list),
        ('SAR_diff',  sar_diff_list),
        ('VH',        vh_list),
        ('VV',        vv_list),
    ]:
        feats.update(temporal_stats(lst, name))
        feats.update(temporal_trend(lst, name))

    # ── Temporal statistics for raw optical bands ─────────────────────────────
    for band in ['blue', 'green', 'red', 'nir', 'nira', 're1', 're2', 're3', 'swir1', 'swir2']:
        cols = [df[f'{band}_{m}'] for m in MONTHS]
        feats.update(temporal_stats(cols, band))

    # ── Cross-index features (aggregate-level) ────────────────────────────────
    ndwi_m  = pd.concat(ndwi_list, axis=1).mean(axis=1)
    mndwi_m = pd.concat(mndwi_list, axis=1).mean(axis=1)
    ndvi_m  = pd.concat(ndvi_list, axis=1).mean(axis=1)
    vh_m    = pd.concat(vh_list, axis=1).mean(axis=1)
    awei_m  = pd.concat(awei_list, axis=1).mean(axis=1)

    feats['water_score']  = ndwi_m + mndwi_m - ndvi_m        # high = water
    feats['water_sar']    = -vh_m + ndwi_m                     # SAR+optical combo
    feats['awei_sar']     = awei_m - vh_m                      # AWEI amplified by SAR
    feats['ndwi_vh_prod'] = ndwi_m * (-vh_m)                  # both high = pond

    # ── SAR seasonality: wet-season (May–Oct) vs dry-season (Nov–Apr) ─────────
    wet_months  = ['05','06','07','08','09','10']
    dry_months  = ['11','12','01','02','03','04']
    wet_vh  = df[[f'VH_{m}' for m in wet_months]].mean(axis=1)
    dry_vh  = df[[f'VH_{m}' for m in dry_months]].mean(axis=1)
    feats['vh_wet_dry_diff'] = wet_vh - dry_vh
    feats['vh_wet_mean']     = wet_vh
    feats['vh_dry_mean']     = dry_vh

    wet_ndwi = pd.concat([ndwi_list[int(m)-1] for m in wet_months], axis=1).mean(axis=1)
    dry_ndwi = pd.concat([ndwi_list[int(m)-1] for m in dry_months], axis=1).mean(axis=1)
    feats['ndwi_wet_dry_diff'] = wet_ndwi - dry_ndwi

    # ── Missing data features (useful for test set) ───────────────────────────
    all_opt = [df[f'blue_{m}'] for m in MONTHS]
    feats['n_missing_optical'] = pd.concat(all_opt, axis=1).isna().sum(axis=1)

    out = pd.DataFrame(feats, index=df.index)
    return out


print('Building features...')
X_train = build_features(train)
X_test  = build_features(test)
y = train['label'].values

print(f'Feature matrix: {X_train.shape}')
print(f'NaN in train features: {X_train.isna().sum().sum()}')
print(f'NaN in test features:  {X_test.isna().sum().sum()}')

## 3. Quick Separability Check

In [ ]:
check_cols = ['VH_mean', 'NDWI_mean', 'MNDWI_mean', 'AWEI_mean', 
              'water_score', 'SAR_diff_mean', 'VH_cv', 'ndwi_vh_prod']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, check_cols):
    for lbl, color, name in [(0,'steelblue','Non-Pond'), (1,'tomato','Pond')]:
        X_train.loc[y==lbl, col].dropna().hist(
            ax=ax, bins=50, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(col, fontsize=10)
    ax.legend(fontsize=8)

plt.suptitle('Feature Separability: Pond vs Non-Pond', fontsize=13)
plt.tight_layout()
plt.show()

# Print mean values for key features
print('\nMean feature values by class:')
print(pd.concat([X_train[check_cols].assign(label=y).groupby('label')[check_cols].mean().T], axis=1))

## 4. LightGBM — 5-Fold CV

In [ ]:
# Fill NaN with median for tree models that need it (LGB is fine but let's be safe)
X_train_filled = X_train.copy()
X_test_filled  = X_test.copy()
for col in X_train.columns:
    med = X_train[col].median()
    X_train_filled[col] = X_train[col].fillna(med)
    X_test_filled[col]  = X_test[col].fillna(med)

lgb_params = {
    'objective':        'binary',
    'metric':           'auc',
    'n_estimators':     3000,
    'learning_rate':    0.01,
    'num_leaves':       127,
    'max_depth':        -1,
    'min_child_samples': 15,
    'feature_fraction': 0.6,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'reg_alpha':        0.05,
    'reg_lambda':       0.1,
    'min_split_gain':   0.001,
    'random_state':     42,
    'n_jobs':           -1,
    'verbose':          -1,
}

X_arr      = X_train_filled.values
X_test_arr = X_test_filled.values

lgb_oof  = np.zeros(len(X_train))
lgb_test = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_arr, y)):
    X_tr, X_val = X_arr[tr_idx], X_arr[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[
            lgb.early_stopping(150, verbose=False),
            lgb.log_evaluation(500)
        ]
    )
    lgb_oof[val_idx] = model.predict_proba(X_val)[:, 1]
    lgb_test += model.predict_proba(X_test_arr)[:, 1] / N_FOLDS

    auc = roc_auc_score(y_val, lgb_oof[val_idx])
    print(f'LGB Fold {fold+1} AUC: {auc:.4f} | best iter: {model.best_iteration_}')

print(f'\nLGB OOF AUC: {roc_auc_score(y, lgb_oof):.4f}')

## 5. CatBoost — 5-Fold CV

In [ ]:
cat_oof  = np.zeros(len(X_train))
cat_test = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_arr, y)):
    X_tr, X_val = X_arr[tr_idx], X_arr[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.01,
        depth=8,
        l2_leaf_reg=3,
        random_strength=1,
        bagging_temperature=0.5,
        border_count=128,
        eval_metric='AUC',
        random_seed=42,
        early_stopping_rounds=150,
        verbose=500,
        task_type='CPU',
    )
    model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        use_best_model=True,
    )
    cat_oof[val_idx] = model.predict_proba(X_val)[:, 1]
    cat_test += model.predict_proba(X_test_arr)[:, 1] / N_FOLDS

    auc = roc_auc_score(y_val, cat_oof[val_idx])
    print(f'CAT Fold {fold+1} AUC: {auc:.4f}')

print(f'\nCAT OOF AUC: {roc_auc_score(y, cat_oof):.4f}')

## 6. XGBoost — 5-Fold CV

In [ ]:
xgb_oof  = np.zeros(len(X_train))
xgb_test = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_arr, y)):
    X_tr, X_val = X_arr[tr_idx], X_arr[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    model = XGBClassifier(
        n_estimators=3000,
        learning_rate=0.01,
        max_depth=7,
        subsample=0.8,
        colsample_bytree=0.6,
        min_child_weight=5,
        reg_alpha=0.05,
        reg_lambda=1.0,
        eval_metric='auc',
        random_state=42,
        n_jobs=-1,
        early_stopping_rounds=150,
        verbosity=0,
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    xgb_oof[val_idx] = model.predict_proba(X_val)[:, 1]
    xgb_test += model.predict_proba(X_test_arr)[:, 1] / N_FOLDS

    auc = roc_auc_score(y_val, xgb_oof[val_idx])
    print(f'XGB Fold {fold+1} AUC: {auc:.4f}')

print(f'\nXGB OOF AUC: {roc_auc_score(y, xgb_oof):.4f}')

## 7. Stacking — Meta-Learner on OOF

In [ ]:
# Stack OOF predictions as meta-features
oof_stack  = np.column_stack([lgb_oof, cat_oof, xgb_oof])
test_stack = np.column_stack([lgb_test, cat_test, xgb_test])

# Print individual model OOF AUCs
for name, preds in [('LGB', lgb_oof), ('CAT', cat_oof), ('XGB', xgb_oof)]:
    print(f'{name} OOF AUC: {roc_auc_score(y, preds):.4f}')

# Find best blend weights by sweep
best_auc, best_w = 0, (0.33, 0.33, 0.34)
for w1 in np.arange(0.0, 1.01, 0.05):
    for w2 in np.arange(0.0, 1.01 - w1, 0.05):
        w3 = 1.0 - w1 - w2
        if w3 < 0: continue
        blend = w1*lgb_oof + w2*cat_oof + w3*xgb_oof
        auc = roc_auc_score(y, blend)
        if auc > best_auc:
            best_auc, best_w = auc, (w1, w2, w3)

print(f'\nBest blend weights: LGB={best_w[0]:.2f} CAT={best_w[1]:.2f} XGB={best_w[2]:.2f}')
print(f'Best blend OOF AUC: {best_auc:.4f}')

# Also try logistic regression meta-learner
meta_oof  = np.zeros(len(y))
meta_test = np.zeros(len(X_test))

for tr_idx, val_idx in skf.split(oof_stack, y):
    meta = LogisticRegression(C=1.0, max_iter=1000)
    scaler = StandardScaler()
    X_m_tr = scaler.fit_transform(oof_stack[tr_idx])
    X_m_val = scaler.transform(oof_stack[val_idx])
    meta.fit(X_m_tr, y[tr_idx])
    meta_oof[val_idx] = meta.predict_proba(X_m_val)[:, 1]
    meta_test += meta.predict_proba(scaler.transform(test_stack))[:, 1] / N_FOLDS

print(f'Meta-LR OOF AUC: {roc_auc_score(y, meta_oof):.4f}')

## 8. Threshold Optimization & Final Submission

In [ ]:
# Choose best prediction (blend vs meta-LR)
w1, w2, w3 = best_w
blended_oof  = w1*lgb_oof  + w2*cat_oof  + w3*xgb_oof
blended_test = w1*lgb_test + w2*cat_test + w3*xgb_test

# Use whichever OOF is better
candidates = {
    'blend':    (blended_oof, blended_test),
    'meta_lr':  (meta_oof,    meta_test),
    'lgb_only': (lgb_oof,     lgb_test),
}

THRESHOLDS = np.arange(0.05, 0.95, 0.005)

results = {}
for name, (oof_p, test_p) in candidates.items():
    auc = roc_auc_score(y, oof_p)
    f1s = [f1_score(y, (oof_p >= t).astype(int)) for t in THRESHOLDS]
    best_f1_idx = np.argmax(f1s)
    results[name] = {
        'auc': auc, 
        'f1': f1s[best_f1_idx], 
        'thr': THRESHOLDS[best_f1_idx],
        'test_p': test_p,
        'oof_p': oof_p
    }
    print(f'{name:12s} | OOF AUC: {auc:.4f} | Best F1: {f1s[best_f1_idx]:.4f} @ thr={THRESHOLDS[best_f1_idx]:.3f}')

# Pick the best by AUC
BEST = max(results, key=lambda k: results[k]['auc'])
print(f'\nUsing: {BEST}')

FINAL_THR  = results[BEST]['thr']
FINAL_PROB = results[BEST]['test_p']
FINAL_PRED = (FINAL_PROB >= FINAL_THR).astype(int)

print(f'Predicted ponds: {FINAL_PRED.sum()} / {len(FINAL_PRED)} ({FINAL_PRED.mean():.2%})')

In [ ]:
# Threshold plot
oof_p = results[BEST]['oof_p']
f1s = [f1_score(y, (oof_p >= t).astype(int)) for t in THRESHOLDS]

plt.figure(figsize=(9, 4))
plt.plot(THRESHOLDS, f1s, lw=2)
plt.axvline(FINAL_THR, color='red', ls='--', label=f'Best thr={FINAL_THR:.3f}')
plt.xlabel('Decision Threshold'); plt.ylabel('F1 Score')
plt.title(f'OOF F1 vs Threshold — {BEST} | Best F1={max(f1s):.4f}')
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Save submission
sub['TargetF1']   = FINAL_PRED
sub['TargetRAUC'] = FINAL_PROB
sub.to_csv('submission_v2.csv', index=False)

print('Saved: submission_v2.csv')
print(sub.head(10))

## 9. Feature Importance

In [ ]:
# Refit LGB on all train data to get stable importances
final_model = lgb.LGBMClassifier(
    **{**lgb_params, 'n_estimators': 1500, 'learning_rate': 0.01}
)
final_model.fit(X_arr, y)

imp = pd.DataFrame({
    'feature': X_train.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False).head(30)

plt.figure(figsize=(10, 9))
sns.barplot(data=imp, x='importance', y='feature', palette='magma')
plt.title('Top 30 Features (LightGBM)')
plt.tight_layout()
plt.show()